# Whole-gene assembly from oligopools (OMEGA)

Routes a codon-optimized library to [OMEGA](https://github.com/RomeroLab/omega), which fragments each full-length gene into short Golden Gate oligos and picks the junction overhangs that maximize predicted assembly fidelity. The oligos ship as one pool and reassemble into complete genes.

Use this when **every member is a complete gene**. The common case is a set of distinct sequences, orthologs, generative designs, or deep multi-mutants, each a full-length protein optimized on its own. Build those with `SequenceSet`, which is what this notebook does. A single-position scan of one long coding sequence also works (swap in `SubstitutionScan`). For single mutants that share one wild-type backbone, `02-tiled-assembly.ipynb` assembles each mutant into a vector carrying the rest of the sequence, which needs far fewer oligos.

OMEGA is a **separate tool under the GPL-3.0 license**; `library_designer` never imports or bundles it. This notebook runs a **separately installed** OMEGA as a subprocess: it writes the inputs, calls OMEGA's command line, and reads back its output tables.

This tutorial runs on the bundled example set. Run the cells top to bottom, and edit the input cell to use your own sequences. Files are written to `out/`.

## Prerequisites

Install OMEGA once, following its README (it ships a conda environment):

```bash
git clone https://github.com/RomeroLab/omega
cd omega && conda env create -f environment.yml   # creates the 'omega' env
```

Then tell this notebook where it lives, either with the variables in the "Point at your OMEGA install" cell below, or by exporting `OMEGA_HOME` (and `OMEGA_PYTHON` if OMEGA is in its own environment) before launching Jupyter.

In [ ]:
! git clone https://github.com/RomeroLab/omega ~/repos/omega
! cd ~/repos/omega && conda env create -f environment.yml   # creates the 'omega' env

## Input

This tutorial builds its library from the bundled example set `examples/example_designs.faa`, a scaffold protein plus five multi-mutant designs. `SequenceSet.from_fasta` loads them, and each becomes one gene, codon-optimized on its own. For your own library, point `from_fasta` at your FASTA of proteins, or pass a `{name: protein}` mapping to `SequenceSet(spec, proteins=...)`, to design orthologs, generative designs, or multi-mutants.

In [ ]:
from library_designer import LibrarySpec, CodonOptimizationParams, SequenceSet, OmegaParams

spec = LibrarySpec(
    name="example_designs",
    optimization=CodonOptimizationParams(species="e_coli"),   # picks each member's codons
    platform="pooled",
)

# Load the bundled example set (or point from_fasta at your own FASTA of proteins).
designs = SequenceSet.from_fasta(spec, "../../examples/example_designs.faa")
{name: f"{len(seq)} aa" for name, seq in designs.proteins.items()}

## Build the codon-optimized library

`generate().codon_optimize()` codon-optimizes each member on its own under the spec's rules, avoiding the BsaI site and Shine-Dalgarno-like motifs. The members are distinct genes, so there is no shared reference sequence. `check()` confirms every member translates back to its protein and carries no forbidden site. OMEGA takes these finished sequences and works out how to build them; it does no codon optimization of its own.

In [ ]:
lib = designs.generate().codon_optimize()
print(len(lib), "full-length genes")
if lib.failed:
    print(len(lib.failed), "could not be optimized:", lib.failed)
print(lib.check())

## Point at your OMEGA install

`omega_home` is your OMEGA checkout (the folder containing `code/omega.py`). `omega_python` is the interpreter of OMEGA's environment; leave it `None` to use plain `python`. Either can instead be supplied through the `OMEGA_HOME` / `OMEGA_PYTHON` environment variables.

In [3]:
omega_home = "~/repos/omega"     # your OMEGA checkout (contains code/omega.py)
omega_python = None               # e.g. "~/miniconda3/envs/omega/bin/python"; None uses "python"

## Preview the FASTA OMEGA will receive

`to_omega_fasta()` writes the coding region of each member. OMEGA adds the Golden Gate recognition sites, backbone overhangs, amplification primers, and padding itself, so the FASTA carries the bare genes only.

In [ ]:
lib.to_omega_fasta("out/omega_input.fasta")
print("".join(open("out/omega_input.fasta").readlines()[:6]))

## Assemble with OMEGA

`assemble_with_omega()` writes the FASTA and a primer CSV, runs OMEGA, and parses its three output tables into a result object. `njunctions` is the Golden Gate site budget per subpool: more sites let longer genes assemble but lower the predicted per-reaction fidelity. OMEGA splits the library across as many subpools as needed and optimizes the junctions within each by simulated annealing.

In [ ]:
result = lib.assemble_with_omega(
    OmegaParams(
        njunctions=40,          # Golden Gate sites per subpool
        enzyme="BsaI",          # Type IIS enzyme
        oligo_len=350,          # your synthesis oligo length
        nopt_steps=1000,        # simulated-annealing steps per run
        nopt_runs=5,            # independent runs; the best is kept
    ),
    omega_home=omega_home,
    omega_python=omega_python,
    primer_source="subramanian2018",   # OMEGA's recommended set, bundled with library_designer
)
result

## Results

Three tables come back: the oligos to order (`result.oligos`), per-gene assembly details with predicted fidelity (`result.genes`), and per-subpool statistics including the random seed OMEGA used (`result.pools`).

In [ ]:
print(len(result.oligos), "oligos  |  ", len(result.genes), "genes  |  ", len(result.pools), "subpools")
result.pools

In [ ]:
result.genes.head()

## Export

Write the OMEGA order tables and the run's design specs into `out/`. The design specs record the OMEGA parameters and the per-subpool seeds, so the assembly can be reproduced.

In [ ]:
import os
os.makedirs("out", exist_ok=True)

result.oligos.to_csv("out/oligo_order.csv", index=False)            # the pooled oligo order
result.genes.to_csv("out/optimization_results.csv", index=False)   # per-gene oligos + fidelity
result.pools.to_csv("out/pool_stats.csv", index=False)             # per-subpool stats + seed
lib.to_full_csv("out/library_full.csv")                            # every gene with QC columns
lib.to_design_specs(f"out/{lib.spec.name}_design_specs.json")      # spec, reference, OMEGA run (params, seeds), versions

for f in sorted(os.listdir("out")):
    print("out/" + f)

## Notes

- OMEGA is installed separately and carries the GPL-3.0 license; `library_designer` (MIT) only shells out to it, loading none of its code. See https://github.com/RomeroLab/omega and Freschlin, Yang & Romero, *bioRxiv* (2025).
- `SequenceSet` optimizes each member independently, because the members are distinct genes. A `SubstitutionScan` instead stamps single mutants onto one shared reference. Use the scan when the library is single mutants of one gene that happens to be too long for a single oligo.
- `njunctions` is the per-subpool Golden Gate budget. The paper assembles up to ~2.6 kb genes using as many as 70 sites per reaction; raise it for longer genes, lower it for higher fidelity.
- Primers reuse the bundled `subramanian2018` set (OMEGA's recommended orthogonal primers). Point `primer_source` at your own CSV to override.
- The FASTA carries coding regions only. Adaptors on the `LibrarySpec` are not sent, OMEGA supplies its own flanking sites and primers.
- For single-mutant libraries that share a wild-type backbone, `02-tiled-assembly.ipynb` is the cheaper route (one variable tile per member into a shared destination vector).